# Advanced Problems with Solutions: `itertools.groupby`

This notebook is a self-contained advanced practice set for Python's `itertools.groupby`.

## What you will practice

- Consecutive grouping vs SQL-style grouping
- Streaming aggregation without loading unnecessary data
- Correct key-function design
- The shared-underlying-iterator caveat
- Nested grouping
- Run-length encoding
- Sessionization
- Streak detection
- Interval merging
- Top-N selection per group
- When **not** to use `groupby`

> Core rule: `itertools.groupby` only groups **adjacent/consecutive** items with the same key.  
> For SQL-like grouping, sort by the same key first.


In [1]:
from __future__ import annotations

import csv
import json
import math
from collections import defaultdict, Counter
from dataclasses import dataclass
from datetime import datetime, date, timedelta
from heapq import nlargest
from io import StringIO
from itertools import groupby, islice
from operator import itemgetter
from pprint import pprint
from typing import Any, Callable, Iterable, Iterator, TypeVar

T = TypeVar("T")
K = TypeVar("K")


## Self-contained practice data

The original lesson used a car CSV file. To make this notebook portable, all examples below use embedded data.


In [2]:
cars_csv = """make,model,year,category
ACURA,ILX,2014,sedan
ACURA,MDX,2014,suv
ACURA,RDX,2014,suv
BMW,320I,2014,sedan
BMW,X5,2014,suv
BMW,M3,2014,coupe
TESLA,MODEL S,2014,sedan
TESLA,MODEL X,2014,suv
TOYOTA,CAMRY,2014,sedan
TOYOTA,COROLLA,2014,sedan
TOYOTA,RAV4,2014,suv
FORD,F-150,2014,truck
FORD,FOCUS,2014,hatchback
FORD,ESCAPE,2014,suv
"""

messy_cars_csv = """make,model,year,category
Acura,ILX,2014,sedan
ACURA ,MDX,2014,suv
aCuRa,RDX,2014,suv
BMW,320I,2014,sedan
B.M.W.,X5,2014,suv
Tesla,Model S,2014,sedan
TESLA INC.,Model X,2014,suv
Toyota,Camry,2014,sedan
TOYOTA,COROLLA,2014,sedan
toyota,RAV4,2014,suv
"""

orders = [
    {"order_id": 1, "region": "EU", "product": "Python Course", "date": "2026-01-05", "amount": 120},
    {"order_id": 2, "region": "US", "product": "Python Course", "date": "2026-01-06", "amount": 150},
    {"order_id": 3, "region": "EU", "product": "Data Course", "date": "2026-01-07", "amount": 200},
    {"order_id": 4, "region": "EU", "product": "Python Course", "date": "2026-02-02", "amount": 130},
    {"order_id": 5, "region": "US", "product": "Data Course", "date": "2026-02-03", "amount": 240},
    {"order_id": 6, "region": "APAC", "product": "Python Course", "date": "2026-02-04", "amount": 110},
    {"order_id": 7, "region": "EU", "product": "Data Course", "date": "2026-02-05", "amount": 210},
    {"order_id": 8, "region": "US", "product": "Python Course", "date": "2026-02-07", "amount": 160},
    {"order_id": 9, "region": "APAC", "product": "Data Course", "date": "2026-03-01", "amount": 190},
    {"order_id": 10, "region": "EU", "product": "Python Course", "date": "2026-03-09", "amount": 140},
]

events = [
    {"user": "ana", "ts": "2026-01-01 09:00", "action": "open"},
    {"user": "bob", "ts": "2026-01-01 09:02", "action": "open"},
    {"user": "ana", "ts": "2026-01-01 09:10", "action": "click"},
    {"user": "ana", "ts": "2026-01-01 10:00", "action": "purchase"},
    {"user": "bob", "ts": "2026-01-01 09:20", "action": "click"},
    {"user": "bob", "ts": "2026-01-01 11:00", "action": "logout"},
    {"user": "ana", "ts": "2026-01-01 10:20", "action": "logout"},
    {"user": "cora", "ts": "2026-01-01 12:00", "action": "open"},
    {"user": "cora", "ts": "2026-01-01 12:05", "action": "click"},
]

habit_log = [
    {"user": "ana", "day": "2026-01-01"},
    {"user": "ana", "day": "2026-01-02"},
    {"user": "ana", "day": "2026-01-03"},
    {"user": "ana", "day": "2026-01-05"},
    {"user": "ana", "day": "2026-01-06"},
    {"user": "bob", "day": "2026-01-01"},
    {"user": "bob", "day": "2026-01-03"},
    {"user": "bob", "day": "2026-01-04"},
    {"user": "bob", "day": "2026-01-05"},
    {"user": "bob", "day": "2026-01-07"},
]

dna = "AAAACCCGGTTTTTAAACCCCCGGGTT"

status_stream = [
    "OK", "OK", "OK",
    "WARN", "WARN",
    "OK",
    "FAIL", "FAIL", "FAIL", "FAIL",
    "OK", "OK",
]

scores = [
    {"student": "Ana", "course": "Python", "score": 91},
    {"student": "Ben", "course": "Python", "score": 88},
    {"student": "Cora", "course": "Python", "score": 95},
    {"student": "Dan", "course": "Python", "score": 72},
    {"student": "Ana", "course": "Data", "score": 89},
    {"student": "Ben", "course": "Data", "score": 94},
    {"student": "Cora", "course": "Data", "score": 85},
    {"student": "Dan", "course": "Data", "score": 98},
    {"student": "Eli", "course": "Data", "score": 91},
]

interval_rows = [
    ("chr1", "gene", 1, 5),
    ("chr1", "gene", 4, 10),
    ("chr1", "gene", 12, 20),
    ("chr1", "exon", 2, 3),
    ("chr1", "exon", 3, 8),
    ("chr2", "gene", 1, 2),
    ("chr2", "gene", 2, 9),
    ("chr2", "gene", 15, 18),
]


## Utility functions


In [3]:
def read_csv_dicts(text: str) -> list[dict[str, str]]:
    """Parse embedded CSV text into a list of dictionaries."""
    return list(csv.DictReader(StringIO(text)))


def materialize_groupby(
    iterable: Iterable[T],
    key: Callable[[T], K] | None = None
) -> list[tuple[K, list[T]]]:
    """
    Safely convert groupby output into reusable groups.

    Why this matters:
    - groupby returns group iterators.
    - Those group iterators share the same underlying source.
    - Once the parent groupby advances, previous groups are no longer available.
    """
    return [(group_key, list(items)) for group_key, items in groupby(iterable, key=key)]


def count_iter(iterator: Iterable[Any]) -> int:
    """Count items from any iterable without requiring len()."""
    return sum(1 for _ in iterator)


def sorted_groupby(
    iterable: Iterable[T],
    key: Callable[[T], K],
) -> Iterator[tuple[K, Iterator[T]]]:
    """SQL-style grouping helper: sort by key, then group by the same key."""
    return groupby(sorted(iterable, key=key), key=key)


# Problem 1 — Consecutive grouping vs SQL-style grouping

## Task

Given a list where repeated values are not necessarily adjacent, write two functions:

1. `consecutive_counts(items)`  
   Counts only consecutive runs.

2. `sql_style_counts(items)`  
   Counts all equal values together, like SQL `GROUP BY`.

## Requirements

- Use `itertools.groupby` in both functions.
- Do not use `Counter` in the solution.
- Include tests that prove the two functions behave differently.


In [4]:
def consecutive_counts(items: Iterable[T]) -> list[tuple[T, int]]:
    """Count adjacent runs only."""
    return [(key, count_iter(group)) for key, group in groupby(items)]


def sql_style_counts(items: Iterable[T]) -> list[tuple[T, int]]:
    """Count all equal items by sorting before grouping."""
    sorted_items = sorted(items)
    return [(key, count_iter(group)) for key, group in groupby(sorted_items)]


sample = ["A", "A", "B", "A", "C", "C", "B", "B"]

print("Consecutive:", consecutive_counts(sample))
print("SQL-style:  ", sql_style_counts(sample))

assert consecutive_counts(sample) == [
    ("A", 2),
    ("B", 1),
    ("A", 1),
    ("C", 2),
    ("B", 2),
]
assert sql_style_counts(sample) == [
    ("A", 3),
    ("B", 3),
    ("C", 2),
]


Consecutive: [('A', 2), ('B', 1), ('A', 1), ('C', 2), ('B', 2)]
SQL-style:   [('A', 3), ('B', 3), ('C', 2)]


# Problem 2 — Streaming counts from already-sorted CSV rows

## Task

The car CSV is already sorted by `make`. Write a function that counts models per make using a streaming approach.

## Requirements

- Parse rows using `csv.DictReader`.
- Use `groupby`.
- Do not build a dictionary of all makes.
- Do not call `len(group)` because group iterators do not implement `len`.


In [5]:
def count_models_by_make_sorted_csv(csv_text: str) -> list[tuple[str, int]]:
    rows = csv.DictReader(StringIO(csv_text))
    groups = groupby(rows, key=itemgetter("make"))
    return [(make, count_iter(models)) for make, models in groups]


make_counts = count_models_by_make_sorted_csv(cars_csv)
pprint(make_counts)

assert make_counts == [
    ("ACURA", 3),
    ("BMW", 3),
    ("TESLA", 2),
    ("TOYOTA", 3),
    ("FORD", 3),
]

# Notice: the function is "streaming", but only correct if rows with the same make are adjacent.


[('ACURA', 3), ('BMW', 3), ('TESLA', 2), ('TOYOTA', 3), ('FORD', 3)]


# Problem 3 — Fix the bug: unsorted CSV produces split groups

## Task

The previous function assumes that all equal keys are adjacent.  
Now write a SQL-style function that works even when the rows are not sorted.

## Requirements

- Sort rows by the same key used for grouping.
- Keep the implementation easy to inspect.
- Return a list of `(make, count)` tuples.


In [6]:
def count_models_by_make_sql_style(csv_text: str) -> list[tuple[str, int]]:
    rows = list(csv.DictReader(StringIO(csv_text)))
    rows.sort(key=itemgetter("make"))
    return [(make, count_iter(models)) for make, models in groupby(rows, key=itemgetter("make"))]


unsorted_cars_csv = """make,model,year,category
BMW,X5,2014,suv
ACURA,ILX,2014,sedan
TOYOTA,RAV4,2014,suv
BMW,M3,2014,coupe
ACURA,MDX,2014,suv
TOYOTA,CAMRY,2014,sedan
"""

print("Wrong if treated as already sorted:")
pprint(count_models_by_make_sorted_csv(unsorted_cars_csv))

print("\nCorrect SQL-style grouping:")
pprint(count_models_by_make_sql_style(unsorted_cars_csv))

assert count_models_by_make_sql_style(unsorted_cars_csv) == [
    ("ACURA", 2),
    ("BMW", 2),
    ("TOYOTA", 2),
]


Wrong if treated as already sorted:
[('BMW', 1),
 ('ACURA', 1),
 ('TOYOTA', 1),
 ('BMW', 1),
 ('ACURA', 1),
 ('TOYOTA', 1)]

Correct SQL-style grouping:
[('ACURA', 2), ('BMW', 2), ('TOYOTA', 2)]


# Problem 4 — Normalize messy grouping keys

## Task

Real data often has inconsistent labels such as:

- `"Acura"`
- `"ACURA "`
- `"aCuRa"`
- `"B.M.W."`
- `"TESLA INC."`

Write a robust make-counting function that normalizes make names before grouping.

## Requirements

- Create a separate `normalize_make` function.
- Normalize case, spaces, periods, and simple aliases.
- Sort using the normalized key before grouping.


In [7]:
def normalize_make(make: str) -> str:
    cleaned = make.strip().upper().replace(".", "")
    cleaned = " ".join(cleaned.split())

    aliases = {
        "TESLA INC": "TESLA",
    }
    return aliases.get(cleaned, cleaned)


def count_models_by_normalized_make(csv_text: str) -> list[tuple[str, int]]:
    rows = list(csv.DictReader(StringIO(csv_text)))
    rows.sort(key=lambda row: normalize_make(row["make"]))
    return [
        (make, count_iter(group))
        for make, group in groupby(rows, key=lambda row: normalize_make(row["make"]))
    ]


normalized_counts = count_models_by_normalized_make(messy_cars_csv)
pprint(normalized_counts)

assert normalized_counts == [
    ("ACURA", 3),
    ("BMW", 2),
    ("TESLA", 2),
    ("TOYOTA", 3),
]


[('ACURA', 3), ('BMW', 2), ('TESLA', 2), ('TOYOTA', 3)]


# Problem 5 — Run-length encoding and decoding

## Task

Run-length encoding compresses consecutive repeated values:

```text
AAAABB -> [('A', 4), ('B', 2)]
```

Write:

1. `rle_encode(iterable)`
2. `rle_decode(pairs)`

## Requirements

- Use `groupby` for encoding.
- Decode into a list.
- Work for strings, lists, and tuples.


In [8]:
def rle_encode(iterable: Iterable[T]) -> list[tuple[T, int]]:
    return [(key, count_iter(group)) for key, group in groupby(iterable)]


def rle_decode(pairs: Iterable[tuple[T, int]]) -> list[T]:
    decoded = []
    for value, count in pairs:
        decoded.extend([value] * count)
    return decoded


encoded_dna = rle_encode(dna)
decoded_dna = "".join(rle_decode(encoded_dna))

print("DNA:", dna)
print("RLE:", encoded_dna)
print("Decoded:", decoded_dna)

assert decoded_dna == dna
assert rle_encode(["OK", "OK", "FAIL", "FAIL", "OK"]) == [
    ("OK", 2),
    ("FAIL", 2),
    ("OK", 1),
]


DNA: AAAACCCGGTTTTTAAACCCCCGGGTT
RLE: [('A', 4), ('C', 3), ('G', 2), ('T', 5), ('A', 3), ('C', 5), ('G', 3), ('T', 2)]
Decoded: AAAACCCGGTTTTTAAACCCCCGGGTT


# Problem 6 — Adjacent status compression with severity

## Task

Given a stream of status values, summarize consecutive runs and enrich each run with severity.

## Requirements

Return dictionaries with:

- `status`
- `count`
- `severity`

Use this severity mapping:

```python
OK -> 0
WARN -> 1
FAIL -> 2
```


In [9]:
SEVERITY = {"OK": 0, "WARN": 1, "FAIL": 2}


def summarize_status_runs(statuses: Iterable[str]) -> list[dict[str, int | str]]:
    result = []
    for status, group in groupby(statuses):
        result.append({
            "status": status,
            "count": count_iter(group),
            "severity": SEVERITY[status],
        })
    return result


status_summary = summarize_status_runs(status_stream)
pprint(status_summary)

assert status_summary == [
    {"status": "OK", "count": 3, "severity": 0},
    {"status": "WARN", "count": 2, "severity": 1},
    {"status": "OK", "count": 1, "severity": 0},
    {"status": "FAIL", "count": 4, "severity": 2},
    {"status": "OK", "count": 2, "severity": 0},
]


[{'count': 3, 'severity': 0, 'status': 'OK'},
 {'count': 2, 'severity': 1, 'status': 'WARN'},
 {'count': 1, 'severity': 0, 'status': 'OK'},
 {'count': 4, 'severity': 2, 'status': 'FAIL'},
 {'count': 2, 'severity': 0, 'status': 'OK'}]


# Problem 7 — Multi-key sales aggregation

## Task

Aggregate orders by `(region, product)`.

For each group, compute:

- number of orders
- total amount
- average amount
- max amount

## Requirements

- Sort by `(region, product)`.
- Group by `(region, product)`.
- Consume each group only once.


In [10]:
def aggregate_sales_by_region_product(order_rows: Iterable[dict[str, Any]]) -> list[dict[str, Any]]:
    keyfunc = lambda row: (row["region"], row["product"])
    sorted_rows = sorted(order_rows, key=keyfunc)

    report = []
    for (region, product), group in groupby(sorted_rows, key=keyfunc):
        group_rows = list(group)
        amounts = [row["amount"] for row in group_rows]
        report.append({
            "region": region,
            "product": product,
            "orders": len(group_rows),
            "total": sum(amounts),
            "average": round(sum(amounts) / len(amounts), 2),
            "max": max(amounts),
        })

    return report


sales_report = aggregate_sales_by_region_product(orders)
pprint(sales_report)

assert sales_report == [
    {"region": "APAC", "product": "Data Course", "orders": 1, "total": 190, "average": 190.0, "max": 190},
    {"region": "APAC", "product": "Python Course", "orders": 1, "total": 110, "average": 110.0, "max": 110},
    {"region": "EU", "product": "Data Course", "orders": 2, "total": 410, "average": 205.0, "max": 210},
    {"region": "EU", "product": "Python Course", "orders": 3, "total": 390, "average": 130.0, "max": 140},
    {"region": "US", "product": "Data Course", "orders": 1, "total": 240, "average": 240.0, "max": 240},
    {"region": "US", "product": "Python Course", "orders": 2, "total": 310, "average": 155.0, "max": 160},
]


[{'average': 190.0,
  'max': 190,
  'orders': 1,
  'product': 'Data Course',
  'region': 'APAC',
  'total': 190},
 {'average': 110.0,
  'max': 110,
  'orders': 1,
  'product': 'Python Course',
  'region': 'APAC',
  'total': 110},
 {'average': 205.0,
  'max': 210,
  'orders': 2,
  'product': 'Data Course',
  'region': 'EU',
  'total': 410},
 {'average': 130.0,
  'max': 140,
  'orders': 3,
  'product': 'Python Course',
  'region': 'EU',
  'total': 390},
 {'average': 240.0,
  'max': 240,
  'orders': 1,
  'product': 'Data Course',
  'region': 'US',
  'total': 240},
 {'average': 155.0,
  'max': 160,
  'orders': 2,
  'product': 'Python Course',
  'region': 'US',
  'total': 310}]


# Problem 8 — Nested grouping: region → month → product

## Task

Create a nested sales summary:

```python
{
    region: {
        month: {
            product: total_amount
        }
    }
}
```

## Requirements

- Use `groupby` at each level.
- Sort by the full nesting key: `(region, month, product)`.
- Do not use `pandas`.


In [11]:
def month_of(order: dict[str, Any]) -> str:
    return order["date"][:7]


def nested_sales_summary(order_rows: Iterable[dict[str, Any]]) -> dict[str, dict[str, dict[str, int]]]:
    sorted_rows = sorted(order_rows, key=lambda row: (row["region"], month_of(row), row["product"]))

    summary: dict[str, dict[str, dict[str, int]]] = {}
    for region, region_rows in groupby(sorted_rows, key=itemgetter("region")):
        summary[region] = {}

        # Important: materialize only this region because we need to group it again.
        region_list = list(region_rows)
        for month, month_rows in groupby(region_list, key=month_of):
            summary[region][month] = {}

            month_list = list(month_rows)
            for product, product_rows in groupby(month_list, key=itemgetter("product")):
                summary[region][month][product] = sum(row["amount"] for row in product_rows)

    return summary


nested = nested_sales_summary(orders)
pprint(nested)

assert nested["EU"]["2026-01"]["Data Course"] == 200
assert nested["EU"]["2026-02"]["Data Course"] == 210
assert nested["US"]["2026-02"]["Python Course"] == 160


{'APAC': {'2026-02': {'Python Course': 110}, '2026-03': {'Data Course': 190}},
 'EU': {'2026-01': {'Data Course': 200, 'Python Course': 120},
        '2026-02': {'Data Course': 210, 'Python Course': 130},
        '2026-03': {'Python Course': 140}},
 'US': {'2026-01': {'Python Course': 150},
        '2026-02': {'Data Course': 240, 'Python Course': 160}}}


# Problem 9 — The shared-iterator caveat

## Task

Demonstrate why this pattern is dangerous:

```python
groups = list(groupby(data, key=...))
```

Then write the safe version.

## Requirements

- Show the broken result.
- Show the fixed result.
- Explain the difference in one sentence.


In [12]:
tuple_data = (
    (1, "abc"),
    (1, "bcd"),
    (2, "pyt"),
    (2, "yth"),
    (2, "tho"),
    (3, "hon"),
)

# Broken pattern:
broken_groups = list(groupby(tuple_data, key=lambda row: row[0]))
broken_result = [(key, list(items)) for key, items in broken_groups]

# Safe pattern:
safe_result = materialize_groupby(tuple_data, key=lambda row: row[0])

print("Broken:", broken_result)
print("Safe:  ", safe_result)

assert broken_result != safe_result
assert safe_result == [
    (1, [(1, "abc"), (1, "bcd")]),
    (2, [(2, "pyt"), (2, "yth"), (2, "tho")]),
    (3, [(3, "hon")]),
]

print("\nExplanation: each group iterator shares the parent source, so advancing the parent can exhaust previous groups.")


Broken: [(1, []), (2, []), (3, [])]
Safe:   [(1, [(1, 'abc'), (1, 'bcd')]), (2, [(2, 'pyt'), (2, 'yth'), (2, 'tho')]), (3, [(3, 'hon')])]

Explanation: each group iterator shares the parent source, so advancing the parent can exhaust previous groups.


# Problem 10 — Top-N records per group

## Task

For each course, return the top `n` students by score.

## Requirements

- Use `groupby`.
- Sort by course before grouping.
- Use `heapq.nlargest` for top-N inside each group.
- Return a dictionary mapping each course to a list of `(student, score)` tuples.


In [13]:
def top_n_students_by_course(rows: Iterable[dict[str, Any]], n: int = 2) -> dict[str, list[tuple[str, int]]]:
    rows_by_course = sorted(rows, key=itemgetter("course"))

    result = {}
    for course, course_rows in groupby(rows_by_course, key=itemgetter("course")):
        top_rows = nlargest(n, course_rows, key=itemgetter("score"))
        result[course] = [(row["student"], row["score"]) for row in top_rows]

    return result


top_students = top_n_students_by_course(scores, n=2)
pprint(top_students)

assert top_students == {
    "Data": [("Dan", 98), ("Ben", 94)],
    "Python": [("Cora", 95), ("Ana", 91)],
}


{'Data': [('Dan', 98), ('Ben', 94)], 'Python': [('Cora', 95), ('Ana', 91)]}


# Problem 11 — Detect consecutive date streaks per user

## Task

Given habit records, find each user's consecutive-day streaks.

## Example

Ana has:

```text
2026-01-01, 2026-01-02, 2026-01-03
2026-01-05, 2026-01-06
```

So Ana has streak lengths `3` and `2`.

## Requirements

- Sort by `(user, day)`.
- Group by user.
- Within each user, use the classic trick: `date_ordinal - index`.
- Return start date, end date, and length for each streak.


In [14]:
def parse_day(s: str) -> date:
    return datetime.strptime(s, "%Y-%m-%d").date()


def streaks_for_dates(days: Iterable[date]) -> list[dict[str, Any]]:
    sorted_days = sorted(set(days))

    streaks = []
    for _, group in groupby(enumerate(sorted_days), key=lambda pair: pair[1].toordinal() - pair[0]):
        pairs = list(group)
        streak_days = [day for _, day in pairs]
        streaks.append({
            "start": streak_days[0].isoformat(),
            "end": streak_days[-1].isoformat(),
            "length": len(streak_days),
        })

    return streaks


def habit_streaks_by_user(records: Iterable[dict[str, str]]) -> dict[str, list[dict[str, Any]]]:
    sorted_records = sorted(records, key=lambda row: (row["user"], row["day"]))

    result = {}
    for user, user_rows in groupby(sorted_records, key=itemgetter("user")):
        days = [parse_day(row["day"]) for row in user_rows]
        result[user] = streaks_for_dates(days)

    return result


streak_report = habit_streaks_by_user(habit_log)
pprint(streak_report)

assert streak_report == {
    "ana": [
        {"start": "2026-01-01", "end": "2026-01-03", "length": 3},
        {"start": "2026-01-05", "end": "2026-01-06", "length": 2},
    ],
    "bob": [
        {"start": "2026-01-01", "end": "2026-01-01", "length": 1},
        {"start": "2026-01-03", "end": "2026-01-05", "length": 3},
        {"start": "2026-01-07", "end": "2026-01-07", "length": 1},
    ],
}


{'ana': [{'end': '2026-01-03', 'length': 3, 'start': '2026-01-01'},
         {'end': '2026-01-06', 'length': 2, 'start': '2026-01-05'}],
 'bob': [{'end': '2026-01-01', 'length': 1, 'start': '2026-01-01'},
         {'end': '2026-01-05', 'length': 3, 'start': '2026-01-03'},
         {'end': '2026-01-07', 'length': 1, 'start': '2026-01-07'}]}


# Problem 12 — User sessionization with time gaps

## Task

Group events into sessions. A new session starts when the gap between consecutive events for the same user is greater than `30` minutes.

## Requirements

- Sort events by `(user, timestamp)`.
- Group first by user.
- Annotate each user's events with a session number.
- Group the annotated events by session number.
- Return session summaries.


In [15]:
def parse_ts(value: str) -> datetime:
    return datetime.strptime(value, "%Y-%m-%d %H:%M")


def annotate_sessions(
    user_events: Iterable[dict[str, str]],
    gap: timedelta,
) -> Iterator[tuple[int, dict[str, str]]]:
    session_id = 1
    previous_ts: datetime | None = None

    for event in user_events:
        current_ts = parse_ts(event["ts"])

        if previous_ts is not None and current_ts - previous_ts > gap:
            session_id += 1

        yield session_id, event
        previous_ts = current_ts


def sessionize_events(
    event_rows: Iterable[dict[str, str]],
    gap_minutes: int = 30,
) -> list[dict[str, Any]]:
    gap = timedelta(minutes=gap_minutes)
    sorted_events = sorted(event_rows, key=lambda row: (row["user"], parse_ts(row["ts"])))

    sessions = []
    for user, user_events in groupby(sorted_events, key=itemgetter("user")):
        annotated = annotate_sessions(user_events, gap)

        for session_id, session_group in groupby(annotated, key=itemgetter(0)):
            session_events = [event for _, event in session_group]
            sessions.append({
                "user": user,
                "session_id": session_id,
                "start": session_events[0]["ts"],
                "end": session_events[-1]["ts"],
                "events": len(session_events),
                "actions": [event["action"] for event in session_events],
            })

    return sessions


session_report = sessionize_events(events, gap_minutes=30)
pprint(session_report)

assert session_report == [
    {"user": "ana", "session_id": 1, "start": "2026-01-01 09:00", "end": "2026-01-01 09:10", "events": 2, "actions": ["open", "click"]},
    {"user": "ana", "session_id": 2, "start": "2026-01-01 10:00", "end": "2026-01-01 10:20", "events": 2, "actions": ["purchase", "logout"]},
    {"user": "bob", "session_id": 1, "start": "2026-01-01 09:02", "end": "2026-01-01 09:20", "events": 2, "actions": ["open", "click"]},
    {"user": "bob", "session_id": 2, "start": "2026-01-01 11:00", "end": "2026-01-01 11:00", "events": 1, "actions": ["logout"]},
    {"user": "cora", "session_id": 1, "start": "2026-01-01 12:00", "end": "2026-01-01 12:05", "events": 2, "actions": ["open", "click"]},
]


[{'actions': ['open', 'click'],
  'end': '2026-01-01 09:10',
  'events': 2,
  'session_id': 1,
  'start': '2026-01-01 09:00',
  'user': 'ana'},
 {'actions': ['purchase', 'logout'],
  'end': '2026-01-01 10:20',
  'events': 2,
  'session_id': 2,
  'start': '2026-01-01 10:00',
  'user': 'ana'},
 {'actions': ['open', 'click'],
  'end': '2026-01-01 09:20',
  'events': 2,
  'session_id': 1,
  'start': '2026-01-01 09:02',
  'user': 'bob'},
 {'actions': ['logout'],
  'end': '2026-01-01 11:00',
  'events': 1,
  'session_id': 2,
  'start': '2026-01-01 11:00',
  'user': 'bob'},
 {'actions': ['open', 'click'],
  'end': '2026-01-01 12:05',
  'events': 2,
  'session_id': 1,
  'start': '2026-01-01 12:00',
  'user': 'cora'}]


# Problem 13 — Merge intervals by chromosome and feature type

## Task

Merge overlapping or touching intervals, but only within the same `(chromosome, feature)` group.

## Example

```text
chr1 gene 1-5
chr1 gene 4-10
```

becomes:

```text
chr1 gene 1-10
```

## Requirements

- Use a dataclass called `Interval`.
- Sort by `(chrom, feature, start, end)`.
- Group by `(chrom, feature)`.
- Merge intervals inside each group.


In [16]:
@dataclass(frozen=True)
class Interval:
    chrom: str
    feature: str
    start: int
    end: int


def merge_one_group(intervals: Iterable[Interval]) -> list[Interval]:
    merged: list[Interval] = []

    for interval in intervals:
        if not merged or interval.start > merged[-1].end:
            merged.append(interval)
        else:
            previous = merged[-1]
            merged[-1] = Interval(
                chrom=previous.chrom,
                feature=previous.feature,
                start=previous.start,
                end=max(previous.end, interval.end),
            )

    return merged


def merge_intervals(rows: Iterable[tuple[str, str, int, int]]) -> list[Interval]:
    intervals = [Interval(*row) for row in rows]
    intervals.sort(key=lambda item: (item.chrom, item.feature, item.start, item.end))

    result: list[Interval] = []
    for _, interval_group in groupby(intervals, key=lambda item: (item.chrom, item.feature)):
        result.extend(merge_one_group(interval_group))

    return result


merged_intervals = merge_intervals(interval_rows)
pprint(merged_intervals)

assert merged_intervals == [
    Interval(chrom="chr1", feature="exon", start=2, end=8),
    Interval(chrom="chr1", feature="gene", start=1, end=10),
    Interval(chrom="chr1", feature="gene", start=12, end=20),
    Interval(chrom="chr2", feature="gene", start=1, end=9),
    Interval(chrom="chr2", feature="gene", start=15, end=18),
]


[Interval(chrom='chr1', feature='exon', start=2, end=8),
 Interval(chrom='chr1', feature='gene', start=1, end=10),
 Interval(chrom='chr1', feature='gene', start=12, end=20),
 Interval(chrom='chr2', feature='gene', start=1, end=9),
 Interval(chrom='chr2', feature='gene', start=15, end=18)]


# Problem 14 — Detect monotonic runs in a numeric series

## Task

Given a sequence of numbers, split it into consecutive runs based on direction:

- increasing
- decreasing
- flat

## Requirements

- Create a helper `direction(a, b)`.
- Convert adjacent pairs into direction records.
- Use `groupby` to group consecutive directions.
- Return readable summaries.


In [17]:
def direction(a: float, b: float) -> str:
    if b > a:
        return "increasing"
    if b < a:
        return "decreasing"
    return "flat"


def monotonic_runs(values: list[float]) -> list[dict[str, Any]]:
    if len(values) < 2:
        return []

    pair_records = []
    for index in range(len(values) - 1):
        pair_records.append({
            "index": index,
            "start_value": values[index],
            "end_value": values[index + 1],
            "direction": direction(values[index], values[index + 1]),
        })

    runs = []
    for run_direction, group in groupby(pair_records, key=itemgetter("direction")):
        pairs = list(group)
        start_index = pairs[0]["index"]
        end_index = pairs[-1]["index"] + 1

        runs.append({
            "direction": run_direction,
            "start_index": start_index,
            "end_index": end_index,
            "values": values[start_index:end_index + 1],
        })

    return runs


series = [1, 2, 3, 3, 3, 2, 1, 5, 8, 8, 7]
runs = monotonic_runs(series)
pprint(runs)

assert runs == [
    {"direction": "increasing", "start_index": 0, "end_index": 2, "values": [1, 2, 3]},
    {"direction": "flat", "start_index": 2, "end_index": 4, "values": [3, 3, 3]},
    {"direction": "decreasing", "start_index": 4, "end_index": 6, "values": [3, 2, 1]},
    {"direction": "increasing", "start_index": 6, "end_index": 8, "values": [1, 5, 8]},
    {"direction": "flat", "start_index": 8, "end_index": 9, "values": [8, 8]},
    {"direction": "decreasing", "start_index": 9, "end_index": 10, "values": [8, 7]},
]


[{'direction': 'increasing',
  'end_index': 2,
  'start_index': 0,
  'values': [1, 2, 3]},
 {'direction': 'flat', 'end_index': 4, 'start_index': 2, 'values': [3, 3, 3]},
 {'direction': 'decreasing',
  'end_index': 6,
  'start_index': 4,
  'values': [3, 2, 1]},
 {'direction': 'increasing',
  'end_index': 8,
  'start_index': 6,
  'values': [1, 5, 8]},
 {'direction': 'flat', 'end_index': 9, 'start_index': 8, 'values': [8, 8]},
 {'direction': 'decreasing',
  'end_index': 10,
  'start_index': 9,
  'values': [8, 7]}]


# Problem 15 — When not to use `groupby`: order-preserving unsorted grouping

## Task

Sometimes data is not sorted and sorting would be too expensive or would change the desired order.  
Write an order-preserving unsorted grouping helper.

## Requirements

- Do **not** use `itertools.groupby`.
- Use a dictionary of lists.
- Preserve first-seen key order.
- Compare the result with `groupby` on the same unsorted data.


In [18]:
def group_unsorted_preserve_order(
    items: Iterable[T],
    key: Callable[[T], K],
) -> list[tuple[K, list[T]]]:
    groups: dict[K, list[T]] = {}

    for item in items:
        groups.setdefault(key(item), []).append(item)

    return list(groups.items())


letters = ["b", "a", "b", "c", "a", "b"]

wrong_for_unsorted = materialize_groupby(letters)
right_for_unsorted = group_unsorted_preserve_order(letters, key=lambda x: x)

print("groupby result:")
pprint(wrong_for_unsorted)

print("\norder-preserving unsorted result:")
pprint(right_for_unsorted)

assert wrong_for_unsorted == [
    ("b", ["b"]),
    ("a", ["a"]),
    ("b", ["b"]),
    ("c", ["c"]),
    ("a", ["a"]),
    ("b", ["b"]),
]
assert right_for_unsorted == [
    ("b", ["b", "b", "b"]),
    ("a", ["a", "a"]),
    ("c", ["c"]),
]


groupby result:
[('b', ['b']),
 ('a', ['a']),
 ('b', ['b']),
 ('c', ['c']),
 ('a', ['a']),
 ('b', ['b'])]

order-preserving unsorted result:
[('b', ['b', 'b', 'b']), ('a', ['a', 'a']), ('c', ['c'])]


# Problem 16 — Streaming group validation

## Task

Write a function that validates whether an iterable is already grouped by a key.

This is useful before using a streaming `groupby` pipeline.

## Requirements

- Return `True` if each key appears in one consecutive block.
- Return `False` if a key disappears and later reappears.
- Use `groupby`.
- Work without sorting.


In [19]:
def is_already_grouped(items: Iterable[T], key: Callable[[T], K]) -> bool:
    seen: set[K] = set()

    for group_key, _ in groupby(items, key=key):
        if group_key in seen:
            return False
        seen.add(group_key)

    return True


assert is_already_grouped(["A", "A", "B", "B", "C"], key=lambda x: x) is True
assert is_already_grouped(["A", "B", "A"], key=lambda x: x) is False

car_rows_sorted = read_csv_dicts(cars_csv)
assert is_already_grouped(car_rows_sorted, key=itemgetter("make")) is True

car_rows_unsorted = read_csv_dicts("""make,model
BMW,X5
ACURA,ILX
BMW,M3
""")
assert is_already_grouped(car_rows_unsorted, key=itemgetter("make")) is False

print("All groupedness checks passed.")


All groupedness checks passed.


# Problem 17 — Mini-capstone: build a full make report

## Task

Build a report from messy CSV data that returns one dictionary per make:

```python
{
    "make": "ACURA",
    "models": ["ILX", "MDX", "RDX"],
    "categories": {"sedan": 1, "suv": 2},
    "model_count": 3
}
```

## Requirements

- Normalize make names.
- Sort by normalized make.
- Group by normalized make.
- Inside each group, sort model names.
- Count categories with `Counter`.
- Return report sorted by descending `model_count`, then make name.


In [20]:
def make_inventory_report(csv_text: str) -> list[dict[str, Any]]:
    rows = list(csv.DictReader(StringIO(csv_text)))
    rows.sort(key=lambda row: normalize_make(row["make"]))

    report = []
    for make, group in groupby(rows, key=lambda row: normalize_make(row["make"])):
        group_rows = list(group)
        models = sorted(row["model"].upper() for row in group_rows)
        categories = Counter(row["category"].strip().lower() for row in group_rows)

        report.append({
            "make": make,
            "models": models,
            "categories": dict(sorted(categories.items())),
            "model_count": len(models),
        })

    report.sort(key=lambda row: (-row["model_count"], row["make"]))
    return report


inventory = make_inventory_report(messy_cars_csv)
pprint(inventory)

assert inventory == [
    {"make": "ACURA", "models": ["ILX", "MDX", "RDX"], "categories": {"sedan": 1, "suv": 2}, "model_count": 3},
    {"make": "TOYOTA", "models": ["CAMRY", "COROLLA", "RAV4"], "categories": {"sedan": 2, "suv": 1}, "model_count": 3},
    {"make": "BMW", "models": ["320I", "X5"], "categories": {"sedan": 1, "suv": 1}, "model_count": 2},
    {"make": "TESLA", "models": ["MODEL S", "MODEL X"], "categories": {"sedan": 1, "suv": 1}, "model_count": 2},
]


[{'categories': {'sedan': 1, 'suv': 2},
  'make': 'ACURA',
  'model_count': 3,
  'models': ['ILX', 'MDX', 'RDX']},
 {'categories': {'sedan': 2, 'suv': 1},
  'make': 'TOYOTA',
  'model_count': 3,
  'models': ['CAMRY', 'COROLLA', 'RAV4']},
 {'categories': {'sedan': 1, 'suv': 1},
  'make': 'BMW',
  'model_count': 2,
  'models': ['320I', 'X5']},
 {'categories': {'sedan': 1, 'suv': 1},
  'make': 'TESLA',
  'model_count': 2,
  'models': ['MODEL S', 'MODEL X']}]


# Extra challenge prompts

Use these as additional practice. Try solving them before looking back at the solved problems above.

1. Modify `rle_encode` so it accepts a `key` function, such as grouping letters case-insensitively.
2. Stream a large sorted CSV and write one output file per group key.
3. Group transactions by `(account_id, transaction_day)` and flag days with total withdrawals over a threshold.
4. For each user session, compute session duration in minutes.
5. Implement `groupby_chunks(iterable, key, chunk_size)` that yields groups but splits very large groups into smaller chunks.
6. Compare a `defaultdict(list)` grouping solution with a `sort + groupby` solution for readability and memory use.
7. Build a CLI-style function that accepts CSV text, a column name, and an aggregation column, then prints a grouped report.
8. Use `groupby` to remove consecutive duplicate words in a sentence.
9. Use `groupby` to find repeated error-code bursts in application logs.
10. Write tests that prove `groupby` does not behave like SQL unless sorted first.

## Best-practice checklist

- Sort before grouping unless you intentionally want consecutive runs.
- Use the same key function for sorting and grouping.
- Keep key functions small, named, and testable.
- Consume each group exactly once unless you materialize it.
- Convert a group to `list(group)` only when you truly need to reuse or inspect all rows.
- Avoid `len(group)` because group iterators usually do not implement `__len__`.
- Use `defaultdict`/`Counter` when unsorted grouping is the real requirement.
- Add assertions for edge cases: empty input, one item, repeated non-adjacent keys, and messy keys.


In [21]:
# Final smoke tests for edge cases

assert consecutive_counts([]) == []
assert sql_style_counts([]) == []
assert rle_encode("") == []
assert rle_decode([]) == []
assert summarize_status_runs([]) == []
assert monotonic_runs([]) == []
assert monotonic_runs([10]) == []
assert group_unsorted_preserve_order([], key=lambda x: x) == []
assert is_already_grouped([], key=lambda x: x) is True

print("All final edge-case smoke tests passed.")


All final edge-case smoke tests passed.
